# 🔬 HHD × PINNs: Hamiltonian Hyperparameter Dynamics for Physics-Informed Neural Networks

**What you'll learn in this notebook:**
1. What PINNs are and how they solve PDEs
2. Why PINN training is notoriously difficult (the loss balancing problem)
3. How HHD (your Hamiltonian Hyperparameter Dynamics algorithm) solves this
4. Side-by-side comparison: fixed-weight PINN vs. HHD-tuned PINN

---

## 📖 Quick Recap: What Is a PINN?

A **Physics-Informed Neural Network** is a neural network trained to satisfy a partial differential equation (PDE). Instead of learning from labelled data, it learns from the *physics itself*.

**The idea:**
- Input: spatial-temporal coordinates $(x, t)$
- Output: the unknown field $u(x, t)$ (e.g., temperature, velocity)
- Loss: how badly the network violates the PDE + boundary/initial conditions

**The loss has multiple competing terms:**
$$\mathcal{L}_{\text{total}} = w_r \cdot \underbrace{\mathcal{L}_r}_{\text{PDE residual}} + w_b \cdot \underbrace{\mathcal{L}_b}_{\text{boundary}} + w_i \cdot \underbrace{\mathcal{L}_i}_{\text{initial cond.}}$$

**The problem:** finding the right weights $w_r, w_b, w_i$ is an unsolved challenge. That's where HHD comes in.

## 1. Setup & Imports

In [ ]:
import sys
import os
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# Add the pinn_overlap/src directory to path
# (In Colab, upload the pinn_overlap folder or clone the repo)
sys.path.insert(0, os.path.join(os.getcwd(), 'pinn_overlap', 'src'))
# If running from within pinn_overlap/notebooks/:
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))
# If running from within pinn_overlap/:
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

from pinn_model import PINN, compute_derivatives_1d
from pinn_loss import PINNLoss, relative_l2_error
from pde_problems import HeatEquation1D, BurgersEquation1D, PoissonEquation2D
from hhd_pinn_trainer import HHDPINNTrainer, BaselinePINNTrainer

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Plot styling
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

print("✅ All imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 2. Our Test Problem: The 1D Heat Equation

The heat equation describes how temperature diffuses along a rod:

$$\frac{\partial u}{\partial t} = \nu \frac{\partial^2 u}{\partial x^2}$$

**Setup:**
- Rod of length 1, with both ends held at 0°C
- Initial temperature: $u(x, 0) = \sin(\pi x)$ — a sine wave
- Over time, the sine wave decays: $u(x, t) = e^{-\nu\pi^2 t} \sin(\pi x)$

Let's visualize the exact solution first — this is what our PINN needs to learn.

In [ ]:
# Create the problem
problem = HeatEquation1D(nu=0.01)

# Visualize the exact solution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
times = [0.0, 0.25, 0.75]

x = torch.linspace(0, 1, 200).unsqueeze(1)

for ax, t_val in zip(axes, times):
    t = torch.ones_like(x) * t_val
    u_exact = problem.exact_solution(x, t)
    ax.plot(x.numpy(), u_exact.numpy(), 'b-', linewidth=2)
    ax.set_title(f't = {t_val}', fontsize=14)
    ax.set_xlabel('x')
    ax.set_ylabel('u(x, t)')
    ax.set_ylim(-0.1, 1.1)

fig.suptitle('Exact Solution: 1D Heat Equation (ν = 0.01)', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

print("The sine wave slowly decays as heat diffuses.")
print("At t=0: peak temperature = 1.0")
print(f"At t=0.75: peak temperature = {float(torch.exp(torch.tensor(-0.01 * np.pi**2 * 0.75))):.4f}")

## 3. How a PINN Works (Under the Hood)

The key insight of PINNs is using **automatic differentiation** to compute PDE residuals.

Let's see this in action — we'll build a small PINN and compute the derivatives:

In [ ]:
# Build a small PINN
model = PINN(input_dim=2, output_dim=1, hidden_layers=[32, 32, 32], activation='tanh')
print(f"PINN architecture: {model}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters())}")

# Create some test points with gradients enabled
x_test = torch.rand(5, 1, requires_grad=True)
t_test = torch.rand(5, 1, requires_grad=True)

# Compute derivatives using autograd
u, u_t, u_x, u_xx = compute_derivatives_1d(model, x_test, t_test)

print("\n--- Autograd Derivatives (5 sample points) ---")
print(f"u    (network output):  {u.detach().numpy().flatten()}")
print(f"u_t  (∂u/∂t):           {u_t.detach().numpy().flatten()}")
print(f"u_x  (∂u/∂x):           {u_x.detach().numpy().flatten()}")
print(f"u_xx (∂²u/∂x²):         {u_xx.detach().numpy().flatten()}")

# PDE residual: r = u_t - ν·u_xx (should be ≈ 0 when PDE is satisfied)
nu = 0.01
residual = u_t - nu * u_xx
print(f"\nPDE residual (u_t - ν·u_xx): {residual.detach().numpy().flatten()}")
print("(These are NOT zero because the network is untrained!)")

## 4. The Loss Balancing Problem (Why PINNs Are Hard)

Here's the crux of the problem. The PINN loss has three terms:

$$\mathcal{L} = w_r \cdot \mathcal{L}_r + w_b \cdot \mathcal{L}_b + w_i \cdot \mathcal{L}_i$$

**If weights are wrong, training fails:**

| Wrong Weights | What Happens |
|:---|:---|
| $w_r \gg w_b, w_i$ | Network satisfies PDE interior but ignores boundary conditions → garbage |
| $w_b \gg w_r, w_i$ | Network nails boundaries but PDE residual is high → not a valid solution |
| All equal ($w_r = w_b = w_i = 1$) | Works okay for simple problems, but fails on nonlinear/stiff PDEs |

**Current solutions are all heuristics** (SoftAdapt, ReLoBRaLo, NTK weighting) with no formal dynamical grounding.

**HHD's solution:** Treat $w_r, w_b, w_i$ as dynamical variables with positions and momenta, and evolve them under Hamilton's equations. The symplectic integrator prevents weight blow-up, and Metropolis accept/reject prevents bad updates.

Let's see this in practice!

## 5. Baseline: Fixed-Weight PINN Training

First, let's train a PINN with **fixed weights** (the standard approach). We use $w_r = w_b = w_i = 1.0$ and Adam optimizer.

In [ ]:
# --- Baseline: Fixed weights ---
torch.manual_seed(42)
baseline_model = PINN(input_dim=2, output_dim=1, hidden_layers=[64, 64, 64], activation='tanh')
problem = HeatEquation1D(nu=0.01)

baseline_trainer = BaselinePINNTrainer(
    model=baseline_model,
    problem=problem,
    n_epochs=300,
    lr=1e-3,
    w_residual=1.0,   # Fixed weight
    w_boundary=1.0,   # Fixed weight
    w_initial=1.0,    # Fixed weight
    n_collocation=1000,
    n_boundary_pts=100,
    n_initial_pts=100,
)

baseline_history = baseline_trainer.train()

## 6. HHD-PINN: Hamiltonian Loss Weight Co-Evolution

Now let's train the **same architecture** but with HHD driving the loss weights.

The key differences:
- **Phase 1 (Adam warmup):** Same as baseline — fixed weights, get θ into a good basin
- **Phase 2 (HMC co-evolution):** Loss weights $(w_r, w_b, w_i)$ become dynamical variables. They carry momentum, evolve via symplectic leapfrog, and are accepted/rejected via Metropolis.

The weights start at $(1, 1, 1)$ and the dynamics figure out where they should go.

In [ ]:
# --- HHD-PINN: Loss weights evolve via Hamiltonian dynamics ---
torch.manual_seed(42)
hhd_model = PINN(input_dim=2, output_dim=1, hidden_layers=[64, 64, 64], activation='tanh')
problem = HeatEquation1D(nu=0.01)

hhd_trainer = HHDPINNTrainer(
    model=hhd_model,
    problem=problem,
    n_warmup=100,      # Phase 1: Adam warmup
    n_hmc=200,         # Phase 2: HMC co-evolution
    step_size=0.003,   # Leapfrog step size ε
    n_leapfrog=3,      # Leapfrog sub-steps L
    mass_theta=1.0,    # Weight inertia
    mass_lambda=5.0,   # Loss weight inertia (higher = more conservative)
    temperature=1e9,   # Always-accept mode (optimization, not sampling)
    lr_adam=1e-3,
    n_collocation=1000,
    n_boundary=100,
    n_initial=100,
)

hhd_history = hhd_trainer.train()

## 7. Results: Baseline vs. HHD-PINN

Now let's compare the two approaches side by side.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# --- Plot 1: Total Loss Convergence ---
ax = axes[0, 0]
ax.semilogy(baseline_history['total_loss'], 'b-', alpha=0.6, label='Baseline (fixed weights)')
ax.semilogy(hhd_history['total_loss'], 'r-', alpha=0.6, label='HHD-PINN')
# Mark phase boundary
n_warmup = hhd_trainer.n_warmup
ax.axvline(n_warmup, color='gray', linestyle='--', alpha=0.5, label=f'HMC starts (epoch {n_warmup})')
ax.set_xlabel('Epoch')
ax.set_ylabel('Total Loss (log scale)')
ax.set_title('Total Loss Convergence')
ax.legend(fontsize=9)

# --- Plot 2: Individual Loss Components ---
ax = axes[0, 1]
ax.semilogy(baseline_history['L_residual'], 'b-', alpha=0.5, label='Baseline L_r')
ax.semilogy(baseline_history['L_boundary'], 'b--', alpha=0.5, label='Baseline L_b')
ax.semilogy(baseline_history['L_initial'], 'b:', alpha=0.5, label='Baseline L_i')
ax.semilogy(hhd_history['L_residual'], 'r-', alpha=0.5, label='HHD L_r')
ax.semilogy(hhd_history['L_boundary'], 'r--', alpha=0.5, label='HHD L_b')
ax.semilogy(hhd_history['L_initial'], 'r:', alpha=0.5, label='HHD L_i')
ax.axvline(n_warmup, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss Component (log scale)')
ax.set_title('Individual Loss Components')
ax.legend(fontsize=7, ncol=2)

# --- Plot 3: Relative L2 Error ---
ax = axes[0, 2]
ax.semilogy(baseline_history['rel_l2_error'], 'b-', linewidth=2, label='Baseline')
ax.semilogy(hhd_history['rel_l2_error'], 'r-', linewidth=2, label='HHD-PINN')
ax.axvline(n_warmup, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('Relative L2 Error')
ax.set_title('Solution Accuracy (vs Exact)')
ax.legend()

# --- Plot 4: Loss Weight Trajectories ---
ax = axes[1, 0]
ax.plot(hhd_history['w_r'], 'r-', linewidth=2, label='$w_r$ (PDE residual)')
ax.plot(hhd_history['w_b'], 'g-', linewidth=2, label='$w_b$ (boundary)')
if any(w != 1.0 for w in hhd_history['w_i']):
    ax.plot(hhd_history['w_i'], 'b-', linewidth=2, label='$w_i$ (initial cond.)')
ax.axvline(n_warmup, color='gray', linestyle='--', alpha=0.5, label=f'HMC starts')
ax.axhline(1.0, color='k', linestyle=':', alpha=0.3, label='Baseline (fixed=1)')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss Weight')
ax.set_title('HHD Loss Weight Trajectories')
ax.legend(fontsize=9)

# --- Plot 5: Learning Rate Trajectory ---
ax = axes[1, 1]
ax.plot(hhd_history['lr'], 'purple', linewidth=2)
ax.axvline(n_warmup, color='gray', linestyle='--', alpha=0.5)
ax.axhline(1e-3, color='k', linestyle=':', alpha=0.3, label='Baseline lr=0.001')
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.set_title('HHD Learning Rate Evolution')
ax.set_yscale('log')
ax.legend()

# --- Plot 6: Acceptance Rate ---
ax = axes[1, 2]
hmc_acc = [a for a, p in zip(hhd_history['acceptance_rate'], hhd_history['phase']) if p == 'hmc']
if hmc_acc:
    ax.plot(range(n_warmup, n_warmup + len(hmc_acc)), hmc_acc, 'orange', linewidth=2)
ax.axhline(0.65, color='green', linestyle='--', alpha=0.5, label='Ideal range')
ax.axhline(0.85, color='green', linestyle='--', alpha=0.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('Cumulative Acceptance Rate')
ax.set_title('HMC Acceptance Rate')
ax.set_ylim(0, 1)
ax.legend()

plt.suptitle('Baseline (Fixed Weights) vs HHD-PINN (Hamiltonian Loss Weight Tuning)', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

# --- Summary ---
baseline_final = baseline_history['rel_l2_error'][-1]
hhd_final = hhd_history['rel_l2_error'][-1]
print(f"\n{'='*50}")
print(f"RESULTS SUMMARY")
print(f"{'='*50}")
print(f"Baseline (fixed w=1) relative L2 error: {baseline_final:.6f}")
print(f"HHD-PINN relative L2 error:             {hhd_final:.6f}")
if hhd_final < baseline_final:
    improvement = (1 - hhd_final / baseline_final) * 100
    print(f"HHD improvement: {improvement:.1f}% lower error")
else:
    print(f"Baseline was better on this run (PINN training is stochastic)")
print(f"\nFinal HHD weights: w_r={hhd_history['w_r'][-1]:.3f}, w_b={hhd_history['w_b'][-1]:.3f}, w_i={hhd_history['w_i'][-1]:.3f}")

## 8. Solution Comparison: Predicted vs. Exact

Let's visually compare the learned solutions against the exact analytical solution.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Evaluate at t=0.5
x_eval = torch.linspace(0, 1, 200).unsqueeze(1)
t_eval = torch.ones_like(x_eval) * 0.5

with torch.no_grad():
    u_exact = problem.exact_solution(x_eval, t_eval).numpy()
    
    baseline_model.eval()
    u_baseline = baseline_model(torch.cat([x_eval, t_eval], dim=1)).numpy()
    
    hhd_model.eval()
    u_hhd = hhd_model(torch.cat([x_eval, t_eval], dim=1)).numpy()

x_np = x_eval.numpy()

# Plot 1: Baseline vs Exact
ax = axes[0]
ax.plot(x_np, u_exact, 'k-', linewidth=2, label='Exact')
ax.plot(x_np, u_baseline, 'b--', linewidth=2, label='Baseline PINN')
ax.set_title('Baseline (fixed weights)')
ax.set_xlabel('x')
ax.set_ylabel('u(x, 0.5)')
ax.legend()

# Plot 2: HHD vs Exact
ax = axes[1]
ax.plot(x_np, u_exact, 'k-', linewidth=2, label='Exact')
ax.plot(x_np, u_hhd, 'r--', linewidth=2, label='HHD-PINN')
ax.set_title('HHD-PINN (evolving weights)')
ax.set_xlabel('x')
ax.set_ylabel('u(x, 0.5)')
ax.legend()

# Plot 3: Error comparison
ax = axes[2]
ax.plot(x_np, np.abs(u_baseline - u_exact), 'b-', linewidth=2, label='|Baseline - Exact|')
ax.plot(x_np, np.abs(u_hhd - u_exact), 'r-', linewidth=2, label='|HHD - Exact|')
ax.set_title('Pointwise Absolute Error')
ax.set_xlabel('x')
ax.set_ylabel('|error|')
ax.legend()
ax.set_yscale('log')

plt.suptitle('Solution Comparison at t = 0.5', fontsize=14)
plt.tight_layout()
plt.show()

## 9. Harder Problem: 1D Burgers' Equation

Burgers' equation adds **nonlinearity** ($u \cdot u_x$), which makes loss balancing much harder:

$$\frac{\partial u}{\partial t} + u \frac{\partial u}{\partial x} = \nu \frac{\partial^2 u}{\partial x^2}$$

The nonlinear term causes the loss landscape to become ill-conditioned — the PDE residual gradients can be orders of magnitude larger than boundary condition gradients. This is exactly the scenario where HHD's inertia and accept/reject mechanism should help most.

In [ ]:
# --- Burgers' Equation: Baseline vs HHD ---
burgers = BurgersEquation1D(nu=0.01 / np.pi)

# Baseline
torch.manual_seed(42)
burgers_baseline = PINN(input_dim=2, output_dim=1, hidden_layers=[64, 64, 64], activation='tanh')
burgers_baseline_trainer = BaselinePINNTrainer(
    model=burgers_baseline, problem=burgers,
    n_epochs=300, lr=1e-3,
    n_collocation=1000, n_boundary_pts=100, n_initial_pts=100,
)
burgers_baseline_hist = burgers_baseline_trainer.train()

# HHD-PINN
torch.manual_seed(42)
burgers_hhd = PINN(input_dim=2, output_dim=1, hidden_layers=[64, 64, 64], activation='tanh')
burgers_hhd_trainer = HHDPINNTrainer(
    model=burgers_hhd, problem=burgers,
    n_warmup=100, n_hmc=200,
    step_size=0.003, n_leapfrog=3,
    mass_theta=1.0, mass_lambda=5.0,
    temperature=1e9, lr_adam=1e-3,
    n_collocation=1000, n_boundary=100, n_initial=100,
)
burgers_hhd_hist = burgers_hhd_trainer.train()

# --- Comparison Plot ---
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

ax = axes[0]
ax.semilogy(burgers_baseline_hist['total_loss'], 'b-', alpha=0.6, label='Baseline')
ax.semilogy(burgers_hhd_hist['total_loss'], 'r-', alpha=0.6, label='HHD-PINN')
ax.axvline(100, color='gray', linestyle='--', alpha=0.5)
ax.set_title("Burgers: Total Loss")
ax.set_xlabel('Epoch'); ax.legend()

ax = axes[1]
ax.semilogy(burgers_baseline_hist['rel_l2_error'], 'b-', linewidth=2, label='Baseline')
ax.semilogy(burgers_hhd_hist['rel_l2_error'], 'r-', linewidth=2, label='HHD')
ax.set_title("Burgers: Relative L2 Error")
ax.set_xlabel('Epoch'); ax.legend()

ax = axes[2]
ax.plot(burgers_hhd_hist['w_r'], 'r-', label='$w_r$')
ax.plot(burgers_hhd_hist['w_b'], 'g-', label='$w_b$')
ax.plot(burgers_hhd_hist['w_i'], 'b-', label='$w_i$')
ax.axhline(1.0, color='k', linestyle=':', alpha=0.3)
ax.axvline(100, color='gray', linestyle='--', alpha=0.5)
ax.set_title("Burgers: HHD Weight Evolution")
ax.set_xlabel('Epoch'); ax.legend()

plt.suptitle("Burgers' Equation: Baseline vs HHD-PINN", fontsize=14)
plt.tight_layout()
plt.show()

print(f"\nBurgers Baseline L2 error: {burgers_baseline_hist['rel_l2_error'][-1]:.6f}")
print(f"Burgers HHD-PINN L2 error: {burgers_hhd_hist['rel_l2_error'][-1]:.6f}")

## 10. Summary & Key Takeaways

### What We Demonstrated

| Aspect | Baseline PINN | HHD-PINN |
|:---|:---|:---|
| **Loss weights** | Fixed at $(1, 1, 1)$ | Evolved via Hamiltonian dynamics |
| **Weight update rule** | None | Symplectic leapfrog + Metropolis accept/reject |
| **Learning rate** | Fixed | Co-evolved alongside loss weights |
| **Formal guarantees** | None | Energy conservation, bounded error |

### Why HHD Is a Natural Fit for PINNs

1. **PINNs have a multi-component loss** — exactly the kind of hyperparameter (loss weights) that benefits from continuous, in-loop tuning
2. **The loss landscape is ill-conditioned** — symplectic integration prevents weight blow-up that heuristics can't guard against
3. **HHD provides momentum** — when weights are moving in a good direction, inertia keeps them going (heuristics have no memory)
4. **Metropolis accept/reject** — bad weight updates are automatically reverted
5. **No new hyperparameters** — unlike SoftAdapt (lookback window) or NTK weighting (update frequency), HHD's parameters are physical (mass, step size) with clear interpretations

### Connection to the Main HHD Paper

The HHD framework was originally developed for general neural network HPO (learning rate, dropout, architecture). This PINN application shows that the *same mathematical framework* — augmented phase space, symplectic leapfrog, Metropolis acceptance — naturally extends to a completely different domain (PDE solving) where the "hyperparameters" are loss-term weights rather than optimizer settings.

This universality is the core strength of the Hamiltonian formulation: **any continuous hyperparameter can be promoted to a dynamical variable.**